# Day 2 · Clustering that survives scrutiny

**Today's goal:** produce clusters *and* the evidence they are real. On real Ethos data you
have no labels, so the second half is not optional — it is the only validation you get.

### Summary

Every clustering algorithm returns clusters. Always. On any input, including pure noise. So
"we found seven groups" is not a finding. The finding is "we found seven groups that
reappear when we resample the data, and each has a coherent clinical profile." Stability is
the claim; the labels are just the vehicle.

### Technique chosen per subtopic

| Subtopic | We use | Mentioned only |
|---|---|---|
| Partitioning | **k-means** on SVD space | k-medoids, k-modes, spectral |
| Density-based | **HDBSCAN** (sklearn >= 1.3) | DBSCAN, OPTICS, mean-shift |
| Choosing k | **silhouette + elbow** | gap statistic, BIC via GMM, eigengap |
| Validation | **bootstrap ARI** | consensus clustering, co-association matrices |
| Probabilistic | *mentioned* | Gaussian mixtures, latent class analysis |

HDBSCAN earns a note: it is the only method here that can answer "this patient belongs to
no group." For claims that is a feature. Most applicants genuinely have no coherent
trajectory, and forcing them into one manufactures signal.

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 150); pd.set_option('display.max_columns', 40)
import ethos_omop as eo

OUT_DIR = "../data_input"   # extract_cohort() output; use "omop_extract" for extract_omop()
DOMAINS = ['Visit', 'Condition', 'Drug', 'Procedure', 'Measurement']

if not os.path.isdir(OUT_DIR):
    raise FileNotFoundError(
        f"{OUT_DIR}/ not found. Run Day 1's eo.extract_cohort(...) (or "
        f"eo.extract_omop(...) with OUT_DIR='omop_extract') first, or "
        f"python make_demo_extract.py for a synthetic stand-in.")

concept  = eo.load_concept(OUT_DIR)
codesets = eo.build_codesets(concept, verbose=False)
events   = eo.tag_events(eo.load_events(OUT_DIR, domains=DOMAINS), codesets)
person   = eo.load_person(OUT_DIR)
truth    = eo.load_truth(OUT_DIR)                   # None against real data

print(f"events {events.shape}   patients {events.person_id.nunique()}")
print(f"codeset coverage: {events.in_codeset.mean():.1%} of events carry a clinical tag")
print(f"truth labels: {'present (DEMO extract)' if truth is not None else 'ABSENT (real data)'}")
events.head(6)

In [ ]:
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

X  = eo.bag_of_codes(events, min_count=20)
Xn = normalize(X.values.astype(float))
NC = min(24, X.shape[1] - 1)
Z  = TruncatedSVD(NC, random_state=0).fit_transform(Xn)

# held-out reference, demo only
y = (truth.set_index('person_id').archetype.reindex(X.index).values
     if truth is not None else None)
print("Z", Z.shape, " reference labels:", "present" if y is not None else "none")

## 1 · Choosing k

Silhouette measures geometric compactness. It does **not** measure clinical usefulness, and
on real data it is often nearly flat. Run it, but never let it decide alone.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

rows = []
for k in range(2, 12):
    km = KMeans(k, n_init=10, random_state=0).fit(Z)
    r = {'k': k, 'inertia': km.inertia_,
         'silhouette': silhouette_score(Z, km.labels_)}
    if y is not None:
        r['ARI_vs_truth'] = adjusted_rand_score(y, km.labels_)
    rows.append(r)
res = pd.DataFrame(rows)
display(res.round(3))

ncol = 3 if y is not None else 2
fig, ax = plt.subplots(1, ncol, figsize=(4.4 * ncol, 3.2))
ax[0].plot(res.k, res.inertia, marker='o');    ax[0].set_title('elbow (inertia)')
ax[1].plot(res.k, res.silhouette, marker='o'); ax[1].set_title('silhouette')
if y is not None:
    ax[2].plot(res.k, res.ARI_vs_truth, marker='o', color='firebrick')
    ax[2].set_title('ARI vs planted truth (DEMO ONLY)')
for a in ax: a.set_xlabel('k')
plt.tight_layout(); plt.show()

Notice how weakly silhouette tracks the ARI curve. **On real Ethos data you will not have
the red panel at all.** That is exactly why the next section exists — stability substitutes
for the ground truth you do not have.

## 2 · HDBSCAN, and its sensitivity to dimensionality

Density-based clustering degrades sharply as dimensions rise, because density itself stops
being estimable. Watch it happen.

In [ ]:
from sklearn.cluster import HDBSCAN

print(f"{'dims':>5} {'min_cluster':>12} {'clusters':>9} {'noise':>7}"
      + (f" {'ARI':>7}" if y is not None else ""))
for nc in [5, 8, 12, min(24, X.shape[1] - 1)]:
    Zi = normalize(TruncatedSVD(nc, random_state=0).fit_transform(Xn))
    for mcs in [20, 40]:
        l  = HDBSCAN(min_cluster_size=mcs, min_samples=5, copy=True).fit(Zi).labels_
        nk = len(set(l)) - (1 if -1 in l else 0)
        extra = f" {adjusted_rand_score(y, l):>7.3f}" if y is not None else ""
        print(f"{nc:>5} {mcs:>12} {nk:>9} {(l == -1).mean():>7.2f}{extra}")

At 24 dimensions HDBSCAN calls almost everything noise; at 5 it works. **Reduce harder for
density methods than for k-means.** This is the kind of thing that silently ruins an
analysis if nobody checks.

In [ ]:
Z5  = normalize(TruncatedSVD(5, random_state=0).fit_transform(Xn))
hdb = HDBSCAN(min_cluster_size=20, min_samples=5, copy=True).fit(Z5)
print(pd.Series(hdb.labels_).value_counts().sort_index().to_string())
print("\n-1 is the 'no cluster' bucket -- a legitimate and informative answer.")

## 3 · Stability — the section that matters

Resample the patients, re-cluster, measure whether the same patients keep landing together.
Adjusted Rand Index between the reference labelling and the bootstrap labelling, restricted
to the resampled rows.

**Rule of thumb:** mean ARI above ~0.75 is solid, 0.60-0.75 suggestive, below 0.60 means you
found noise and should say so out loud.

In [ ]:
def stability(Z, k, n_boot=30, seed=0):
    rng  = np.random.default_rng(seed)
    base = KMeans(k, n_init=10, random_state=0).fit(Z).labels_
    out  = []
    for b in range(n_boot):
        idx = rng.choice(len(Z), len(Z), replace=True)
        lab = KMeans(k, n_init=10, random_state=b).fit(Z[idx]).labels_
        out.append(adjusted_rand_score(base[idx], lab))
    return np.array(out)

rows = [{'k': k, **dict(zip(['stability_mean', 'stability_sd'],
                            [stability(Z, k).mean(), stability(Z, k).std()]))}
        for k in [4, 5, 6, 7, 8, 9]]
stab = pd.DataFrame(rows)
display(stab.round(3))

plt.figure(figsize=(7, 3))
plt.errorbar(stab.k, stab.stability_mean, yerr=stab.stability_sd, marker='o', capsize=4)
plt.axhline(0.75, ls='--', c='green',  label='solid')
plt.axhline(0.60, ls='--', c='orange', label='suggestive')
plt.xlabel('k'); plt.ylabel('bootstrap ARI'); plt.legend()
plt.title('cluster stability -- the substitute for ground truth')
plt.tight_layout(); plt.show()

## 4 · Profiling — turning a cluster number into a sentence

A cluster is useless until it has a description. Compute each tag's prevalence inside the
cluster versus the cohort overall, and read the top lifts.

In [ ]:
K   = 7
lab = KMeans(K, n_init=10, random_state=0).fit(Z).labels_

prof = X.groupby(lab).mean()
lift = (prof + 0.01).div(X.mean() + 0.01)

summary = []
for c in range(K):
    top = lift.loc[c].sort_values(ascending=False).head(5)
    summary.append({'cluster': c, 'n': int((lab == c).sum()),
                    'signature': ', '.join(f"{i}({v:.0f}x)" for i, v in top.items())})
display(pd.DataFrame(summary))

In [ ]:
# Two honesty checks that work WITHOUT labels.

# (a) does the cluster differ on things you did not cluster on?
win = eo.observation_window(events).set_index('person_id').reindex(X.index)
chk = pd.DataFrame({'cluster': lab,
                    'lookback_yrs': (win.lookback_days / 365.25).values,
                    'n_events': events.groupby('person_id').size().reindex(X.index).values})
display(chk.groupby('cluster').mean().round(2))
print("If clusters differ mainly on n_events, you found UTILISATION, not phenotype.")

In [ ]:
# (b) demo only: how do the clusters line up with the planted archetypes?
if truth is not None:
    display(pd.crosstab(lab, y))
    print("\nCollapsed pairs are the interesting failures -- look for the two diabetes groups.")
else:
    print("Real extract: no labels to cross-tabulate. Stability and (a) are your evidence.")

**The expected demo finding:** stable and progressive diabetes land in the same cluster.
Their code sets are nearly identical; only the *sequence* differs. Bag-of-codes clustering
cannot separate them no matter how you tune k.

That is not a tuning problem. It is a representation problem, and it is tomorrow's subject.

## 5 · Mentioned, not used

- **Gaussian mixtures** give soft membership probabilities, far easier to threshold into a
  rule than a hard label. `sklearn.mixture.GaussianMixture`; use BIC to pick components.
- **Latent class analysis** is the same idea in epidemiology's dialect, and reviewers in
  your domain recognise the name. Worth using for anything you plan to defend.
- **Consensus clustering** aggregates many runs into a co-association matrix and clusters
  that. Slower, noticeably more stable.

## Exercises

1. Re-extract with a different patient sample and re-run. Do the signatures survive?
2. Restrict to `lookback_days >= 730` and repeat. Which clusters were left-truncation
   artifacts?
3. Fit a `GaussianMixture` at k=7 and compare its BIC curve to the silhouette curve.
4. For the cluster that merges the diabetes groups, list the tags that *should* separate
   them. Why does prevalence alone fail?

## Carry forward

The observation that two clinically opposite groups are geometrically inseparable in a bag
of codes.